# Descarga nacional de clima NASA POWER (2003–2025)

Este notebook genera un único dataset climático municipal para México. Consulta NASA POWER una vez por celda meteorológica, conserva JSON reanudables y produce `clima_municipal_2003_2025.csv`.

In [2]:
import logging

import pandas as pd

from proyecto_agronomia.config import (
    NASA_POWER_ANIO_FINAL,
    NASA_POWER_ANIO_INICIAL,
    NASA_POWER_PROCESSED_DIR,
)
from proyecto_agronomia.datasets.clima_nasa import descargar_clima_nasa_por_celdas
from proyecto_agronomia.datasets.inegi import obtener_municipios_inegi
from proyecto_agronomia.processing.integracion_climatica import (
    puntos_representativos_municipales,
)

ANIO_INICIAL = NASA_POWER_ANIO_INICIAL
ANIO_FINAL = NASA_POWER_ANIO_FINAL
RUTA_SALIDA = NASA_POWER_PROCESSED_DIR / "clima_municipal_2003_2025.csv"
logging.basicConfig(level=logging.INFO)
pd.set_option("display.max_columns", None)

## 1. Preparar los puntos municipales de INEGI

In [3]:
gdf_municipios_inegi = obtener_municipios_inegi()
df_puntos_nasa = puntos_representativos_municipales(gdf_municipios_inegi)

df_municipios_nasa = (
    df_puntos_nasa[["cvegeo", "nomgeo", "latitud", "longitud"]]
    .rename(columns={
        "nomgeo": "Nommunicipio",
        "latitud": "Latitud",
        "longitud": "Longitud",
    })
    .sort_values("cvegeo")
    .reset_index(drop=True)
)

assert df_municipios_nasa["cvegeo"].is_unique
assert df_municipios_nasa[["Latitud", "Longitud"]].notna().all().all()
print(f"Municipios preparados: {len(df_municipios_nasa):,}")
display(df_municipios_nasa.head())

2026-10-29 15:59:04.299 | INFO     | proyecto_agronomia.datasets.inegi:obtener_municipios_inegi:29 - Cargando municipios INEGI desde archivo local.
Municipios preparados: 2,478


,cvegeo,Nommunicipio,Latitud,Longitud
0,01001,Aguascalientes,21.848487,-102.305449
1,01002,Asientos,22.132227,-102.095219
2,01003,Calvillo,21.911174,-102.713168
3,01004,Cosío,22.371703,-102.311445
4,01005,Jesús María,21.928961,-102.461647


## 2. Descargar NASA POWER 2003–2025

El archivo completo ya existe, por lo que la bandera permanece desactivada. Cambiarla a `True` reconstruye la descarga nacional. Los JSON por celda permiten reanudar una interrupción.

In [4]:
EJECUTAR_DESCARGA = False

if EJECUTAR_DESCARGA:
    df_clima_nasa = descargar_clima_nasa_por_celdas(
        municipios=df_municipios_nasa,
        inicio=ANIO_INICIAL,
        fin=ANIO_FINAL,
    )
    print(f"Registros descargados: {len(df_clima_nasa):,}")
else:
    print("Descarga omitida; se utilizará el CSV existente.")

Descarga omitida; se utilizará el CSV existente.


## 3. Cargar y validar el dataset completo

In [5]:
if not RUTA_SALIDA.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_SALIDA}. Activa EJECUTAR_DESCARGA para generarlo."
    )

df_clima_nasa = pd.read_csv(RUTA_SALIDA, dtype={"cvegeo": "string"})
df_clima_nasa["cvegeo"] = df_clima_nasa["cvegeo"].str.zfill(5)

claves_esperadas = set(df_municipios_nasa["cvegeo"])
anios_esperados = set(range(ANIO_INICIAL, ANIO_FINAL + 1))
filas_esperadas = len(claves_esperadas) * len(anios_esperados)

assert not df_clima_nasa.duplicated(["cvegeo", "anio"]).any()
assert set(df_clima_nasa["cvegeo"]) == claves_esperadas
assert set(df_clima_nasa["anio"]) == anios_esperados
assert len(df_clima_nasa) == filas_esperadas
assert df_clima_nasa[[
    "precipitacion_anual_mm", "temperatura_media_c"
]].notna().all().all()

resumen = df_clima_nasa.groupby("anio").agg(
    municipios=("cvegeo", "nunique"),
    precipitacion_disponible=("precipitacion_anual_mm", "count"),
    temperatura_disponible=("temperatura_media_c", "count"),
)
display(resumen)
display(df_clima_nasa.head())
print(f"Archivo validado: {RUTA_SALIDA}")
print(f"Periodo: {df_clima_nasa['anio'].min()}-{df_clima_nasa['anio'].max()}")
print(f"Municipios: {df_clima_nasa['cvegeo'].nunique():,}")
print(f"Filas: {len(df_clima_nasa):,}/{filas_esperadas:,}")

,municipios,precipitacion_disponible,temperatura_disponible
anio,,,
2003,2478,2478,2478
2004,2478,2478,2478
2005,2478,2478,2478
2006,2478,2478,2478
2007,2478,2478,2478
2008,2478,2478,2478
2009,2478,2478,2478
2010,2478,2478,2478
2011,2478,2478,2478


,cvegeo,anio,precipitacion_anual_mm,temperatura_media_c,dias_precipitacion,dias_temperatura,fuente
0,01001,2003,642.67,16.815123,365,365,NASA POWER
1,01001,2004,704.01,16.206148,366,366,NASA POWER
2,01001,2005,416.74,16.975014,365,365,NASA POWER
3,01001,2006,595.96,17.130575,365,365,NASA POWER
4,01001,2007,539.70,16.791644,365,365,NASA POWER


Archivo validado: B:\MCD2\JW\agro\proyecto_agronomia\data\processed\clima_nasa\clima_municipal_2003_2025.csv
Periodo: 2003-2025
Municipios: 2,478
Filas: 56,994/56,994
